### Mental model
A tool is a Python function wrapped with metadata. Its name, description, and input schema are sent to the model. The model chooses whether to call it; your application executes it and returns the result.
A strong tool has a clear single-purpose name, type hints for every model-visible argument, and a concise docstring explaining when to use it.

In [1]:
# Run once in a fresh notebook environment
%pip install -U langchain langchain-openai langgraph pydantic


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.9
    Uninstalling langgraph-1.2.9:
      Successfully uninstalled langgraph-1.2.9


In [11]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    api_key = os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
    print(f"OpenAI API key set in environment variable OPENAI_API_KEY: {api_key[0:4]}")

OpenAI API key set in environment variable OPENAI_API_KEY: sk-p


### Stage 1 — Create the smallest useful tool
@tool converts the function into a LangChain tool. Type hints form the schema; the docstring becomes guidance for the model. Call .invoke() when testing a tool directly.

In [12]:
from langchain.tools import tool

@tool
def greet(name: str) -> str:
    """Greets a person with their name."""
    return f"Hello, {name}!"

print(greet.name)
print(greet.description)

greet
Greets a person with their name.


### Stage 2 — Inputs, defaults, and custom metadata
Use snake_case names. Defaults become optional fields in the tool schema. You can override the name and description when the Python function name is not ideal for the model.

In [13]:
from langchain.tools import tool

@tool("search_faq",description="Search the product FAQ.Use for product-policy questions.")
def search(query:str,limit:int = 3) -> str:
    """search FAQ entries"""
    return f"Returning {limit} FAQ results for query: {query}"

print(search.name)
print(search.args)
print(search.description)
search.invoke({"query": "How do I reset my password?", "limit": 5})
    

search_faq
{'query': {'title': 'Query', 'type': 'string'}, 'limit': {'default': 3, 'title': 'Limit', 'type': 'integer'}}
Search the product FAQ.Use for product-policy questions.


'Returning 5 FAQ results for query: How do I reset my password?'

### Stage 3 — Use Pydantic for a richer input schema
For more complex parameters, use a Pydantic model. Field descriptions are especially valuable: they tell the model what values are appropriate.

In [14]:
from typing import Optional
from pydantic import BaseModel
from langchain.tools import tool

class WeatherInput(BaseModel):
    location: str
    unit: Optional[str] = "celsius"

@tool(args_schema=WeatherInput, description="Get the current weather for a given location.")
def get_weather(location: str, unit: Optional[str] = "celsius") -> str:
    """Get the current weather for a given location."""
    return f"Current weather in {location} is 25 degrees {unit}."

print(get_weather.name)
print(get_weather.args)
print(get_weather.description)
get_weather.invoke({"location": "New York", "unit": "fahrenheit"})

get_weather
{'location': {'title': 'Location', 'type': 'string'}, 'unit': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': 'celsius', 'title': 'Unit'}}
Get the current weather for a given location.


'Current weather in New York is 25 degrees fahrenheit.'

### Stage 4 — Return values and errors
Tools usually return strings, but can return JSON-like objects when structured output is more useful. Validate inputs and return a helpful message for expected problems; reserve exceptions for unexpected failures. Never use eval on model-provided text.

In [15]:
@tool
def lookup_order(order_id: str) -> dict:
    """Look up an order by its ID (demo data)."""
    orders = {"A-100": {"status": "shipped", "eta_days": 2}}
    if order_id not in orders:
        return {"found": False, "message": f"No order found for {order_id}"}
    return {"found": True, "order_id": order_id, **orders[order_id]}

lookup_order.invoke({"order_id": "A-100"})

{'found': True, 'order_id': 'A-100', 'status': 'shipped', 'eta_days': 2}

### Stage 5 — Give tools to a model
bind_tools() exposes schemas to a chat model. The model response may contain tool_calls; the application or agent is responsible for executing them. Set OPENAI_API_KEY before running this stage.

In [16]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model_name="gpt-4.1-mini", temperature=0)
model_with_tools = model.bind_tools([greet, search, get_weather, lookup_order])
response = model_with_tools.invoke("Hello! Can you greet me and also tell me the weather in London?")
response.tool_calls

[{'name': 'greet',
  'args': {'name': 'User'},
  'id': 'call_npJgtyqYcCK85SQeH75g2XYL',
  'type': 'tool_call'},
 {'name': 'get_weather',
  'args': {'location': 'London'},
  'id': 'call_sW5Y8KcRRt7azs2ImKf5gMtH',
  'type': 'tool_call'}]

### ToolRuntime: runtime data for tools
ToolRuntime is the current, unified interface for data injected into a tool at execution time. It is hidden from the model's tool schema, so it is appropriate for trusted application data and framework services.
It provides: state, context, store, stream_writer, execution_info, server_info, config, and tool_call_id. The names runtime and config are reserved—do not use them for model-visible arguments.

In [17]:
from dataclasses import dataclass
from typing import Any
from langchain.tools import ToolRuntime

# `runtime` is automatically supplied when the tool runs inside an agent/graph.
# It should not be supplied in .invoke({...}) by the model.

### Runtime topic 1 — State: short-term conversation memory
runtime.state contains the current graph state, including conversation messages and any custom state fields. It lasts for the current conversation/run. Use Command when a tool must update state.

In [20]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langgraph.types import Command

@tool
def last_user_message(runtime:ToolRuntime) -> str:
    """Read the most recent user message from the conversation history."""
    for message in reversed(runtime.state["messages"]):
        if isinstance(message, HumanMessage):
            return message.content
    return "No user message found."

@tool
def set_preferred_language(language: str, runtime: ToolRuntime) -> Command:
    """Save the preferred response language in this conversation's state."""
    return Command(update={
        "preferred_language": language,
        "messages": [ToolMessage(
            content=f"Language set to {language}.",
            tool_call_id=runtime.tool_call_id,
        )],
    })
    
model_with_tools = model.bind_tools([last_user_message, set_preferred_language])
model_with_tools.invoke("Please set my preferred language to Spanish.")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 78, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c31e0081d1', 'id': 'chatcmpl-E7my25ve9ujIP55qtVY1uzZhmQr1E', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fb998-eb0d-7171-ba7c-b07621b8a4e3-0', tool_calls=[{'name': 'set_preferred_language', 'args': {'language': 'Spanish'}, 'id': 'call_7VbaLwTEsFJZ7nYtHlou7QJp', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 16, 'total_tokens': 94, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': 

### Runtime topic 2 — Store: long-term memory
runtime.store is a persistent namespace/key store. Unlike state, its values can survive across conversations. InMemoryStore is useful for learning; choose a durable store (for example Postgres, MongoDB, or Redis) for production.

In [ ]:
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent

@tool
def save_preference(user_id: str, preference: dict[str, Any], runtime: ToolRuntime) -> str:
    """Save a user's preference (demo)."""
    runtime.store.put(("preferences",), user_id, preference)
    return "Preference saved."

@tool
def get_preference(user_id: str, runtime: ToolRuntime) -> dict:
    """Get a user's saved preference (demo)."""
    item = runtime.store.get(("preferences",), user_id)
    return item.value if item else {"found": False}

# store = InMemoryStore()
# agent = create_agent(model, [save_preference, get_preference], store=store)

### Runtime topic 3 — Stream writer: live progress
runtime.stream_writer emits custom progress updates during a long-running tool. It only works inside a LangGraph execution context; use the agent/graph streaming API to receive the updates.

In [25]:
@tool
def import_records(source: str, runtime: ToolRuntime) -> str:
    """Import records from a source (demonstration)."""
    runtime.stream_writer(f"Connecting to {source}...")
    runtime.stream_writer("Validating records...")
    runtime.stream_writer("Import complete.")
    return "Imported 42 records."

# Receive updates with an agent/graph .stream(...) call; do not directly invoke this tool.

### Runtime topic 4 — Execution info, server info, config, and tool-call ID
execution_info exposes the thread ID, run ID, and node attempt for logging or retry-aware behavior. server_info contains assistant, graph, and authenticated-user metadata only on LangGraph Server (otherwise it is None). config carries runnable callbacks/tags/metadata. tool_call_id links an output ToolMessage to the precise model request.
execution_info and server_info require langgraph>=1.1.5 (or deepagents>=0.5.0).

In [26]:
@tool
def log_runtime_metadata(runtime: ToolRuntime) -> str:
    """Log safe runtime metadata for diagnostics."""
    info = runtime.execution_info
    print(f"thread={info.thread_id}, run={info.run_id}, attempt={info.node_attempt}")
    print(f"tool_call_id={runtime.tool_call_id}")
    print(f"tags={runtime.config.get('tags', [])}")

    server = runtime.server_info
    if server is not None:
        print(f"assistant={server.assistant_id}, graph={server.graph_id}")
        if server.user is not None:
            print(f"authenticated_user={server.user.identity}")
    return "Runtime metadata logged."